# 第6回 ニューラルネットワークの小さな計算

目的：順方向計算、重みの1回更新、RNNの状態、SOMの近傍更新を区別して観察します。
外部データ・乱数は使いません。同梱 `.py`・JSONの読み込みも不要です。

1. 授業用venvのPythonをカーネルに選びます。
2. カーネルを再起動します。
3. 先頭から全セルを実行します。
4. 基準出力と途中の値を確認してから、後半の実験セルだけを変更します。

CLIと同じ固定入力を使います。保存・環境準備・結果の読み方は[第6回プログラムの使い方](https://nshrhm.github.io/intro-ai/chap05/programs/guide.html)を参照してください。ZIPでは同梱の `guide.html` も読めます。リンク先はネット接続が必要です。

基準結果を残し、実行前に予想→一つ変更→比較→根拠を説明、の順に進めます。元に戻すときはカーネルを再起動して全セルを実行します。

In [ ]:
import sys
print(sys.executable)
print(sys.version)

## 計算の定義

このセルに全処理を収録しています。まず変更せずに実行してください。
`neuron` は合計の後にしきい値を一度だけ引きます。`chain_step` は更新前の値で二つの勾配を求めます。
`recurrent_states` は呼び出しごとに初期状態へ戻ります。`som_step` は元のベクトルを変更せず、新しい値を返します。

In [ ]:
import math


INPUTS = ((0, 0), (0, 1), (1, 0), (1, 1))
HIDDEN_WEIGHTS = ((-2, 3), (-2, 1))
HIDDEN_THRESHOLDS = (-1, 0.5)
OUTPUT_WEIGHTS = (-60, 94)
OUTPUT_THRESHOLD = -1


def step(u):
    """本教材では境界u=0も1とする。"""
    return int(u >= 0)


def sigmoid(u):
    """大きな正負の値でもexpがオーバーフローしない形。"""
    if u >= 0:
        return 1 / (1 + math.exp(-u))
    e = math.exp(u)
    return e / (1 + e)


def relu(u):
    return max(0, u)


def softmax(scores):
    """最大値を引いてから指数関数を計算する。空列は受け付けない。"""
    if not scores:
        raise ValueError("scores must not be empty")
    maximum = max(scores)
    exponentials = [math.exp(score - maximum) for score in scores]
    total = sum(exponentials)
    return [value / total for value in exponentials]


def neuron(inputs, weights, threshold):
    """重み付き和を完成させてから、しきい値を一度だけ引く。"""
    if len(inputs) != len(weights):
        raise ValueError("input and weight lengths differ")
    u = sum(x * w for x, w in zip(inputs, weights)) - threshold
    return u, step(u)


def forward(inputs, hidden_thresholds=HIDDEN_THRESHOLDS):
    """固定の2入力・中間2個・出力1個。途中の値も返す（学習なし）。"""
    if len(inputs) != 2 or len(hidden_thresholds) != 2:
        raise ValueError("this teaching network needs two inputs and thresholds")
    hidden_pairs = [neuron(inputs, weights, threshold)
                    for weights, threshold in zip(HIDDEN_WEIGHTS, hidden_thresholds)]
    hidden = [pair[1] for pair in hidden_pairs]
    output_u, output = neuron(hidden, OUTPUT_WEIGHTS, OUTPUT_THRESHOLD)
    return {"input": list(inputs), "hidden_u": [pair[0] for pair in hidden_pairs],
            "hidden": hidden, "output_u": output_u, "output": output}


def gradient_step(x=2, target=3, weight=2, rate=0.125):
    """o=w*x, L=(o-target)^2/2 の1回の勾配降下（バイアスなし）。"""
    output = weight * x
    gradient = (output - target) * x
    new_weight = weight - rate * gradient
    new_output = new_weight * x
    return {"weight_before": weight, "output_before": output,
            "loss_before": (output - target) ** 2 / 2,
            "gradient": gradient, "rate": rate, "weight_after": new_weight,
            "output_after": new_output, "loss_after": (new_output - target) ** 2 / 2}


def chain_step(x=1, target=4, a=2, b=3, rate=0.125):
    """h=a*x, o=b*h。両勾配を更新前のa,bから求めた後、同時更新する。"""
    hidden = a * x
    output = b * hidden
    error = output - target
    grad_b = error * hidden
    grad_a = error * b * x
    new_a, new_b = a - rate * grad_a, b - rate * grad_b
    new_output = new_b * (new_a * x)
    return {"hidden_before": hidden, "output_before": output,
            "loss_before": error ** 2 / 2, "grad_a": grad_a, "grad_b": grad_b,
            "a_after": new_a, "b_after": new_b, "output_after": new_output,
            "loss_after": (new_output - target) ** 2 / 2}


def recurrent_states(sequence, recurrent_weight=0.5, initial_state=0):
    """h_t=ReLU(x_t+r*h_(t-1))。呼び出すたび初期状態から開始する。"""
    state = initial_state
    states = []
    for x in sequence:
        state = relu(x + recurrent_weight * state)
        states.append(state)
    return states


def som_step(sample, prototypes, rate=0.5, neighbor_factor=0.5):
    """1次元の地図を1回更新。勝者と隣接ノードのみ入力へ近づける。

    入力空間は任意次元。地図上の隣接はリストの番号の差が1。
    同距離なら先の番号を勝者とする。元の配列を変更しない。
    """
    if not sample or not prototypes or any(len(p) != len(sample) for p in prototypes):
        raise ValueError("nonempty vectors of the same dimension are required")
    if not 0 <= rate <= 1 or not 0 <= neighbor_factor <= 1:
        raise ValueError("rate and neighbor_factor must be between 0 and 1")
    distances = [sum((x - w) ** 2 for x, w in zip(sample, p)) for p in prototypes]
    winner = min(range(len(prototypes)), key=lambda i: distances[i])
    updated = []
    for index, prototype in enumerate(prototypes):
        factor = 1 if index == winner else neighbor_factor if abs(index - winner) == 1 else 0
        updated.append([w + rate * factor * (x - w) for x, w in zip(sample, prototype)])
    return {"winner": winner, "squared_distances": distances, "updated": updated}


def demo_result():
    """照合用。超越関数の結果だけ小数6桁へ丸めて表示する。"""
    return {
        "forward": [forward(x) for x in INPUTS],
        "activations": [{"u": u, "step": step(u), "sigmoid": round(sigmoid(u), 6),
                         "relu": relu(u)} for u in (-2, 0, 2)],
        "softmax_equal": softmax([0, 0]),
        "gradient": gradient_step(),
        "gradient_large_rate": gradient_step(rate=1),
        "chain": chain_step(),
        "rnn_10": recurrent_states([1, 0]),
        "rnn_01": recurrent_states([0, 1]),
        "rnn_no_recurrence": recurrent_states([1, 0], recurrent_weight=0),
        "som": som_step([1, 0], [[0, 0], [2, 2], [4, 0]]),
    }

## 基準結果

4入力 `(0,0), (0,1), (1,0), (1,1)` に対する出力は `0,1,1,0`。
重みは固定なので、学習した結果ではありません。
1回の勾配更新では損失が `0.5 → 0.125`。RNNは `[1,0] → [1,0.5]`、`[0,1] → [0,1]`。
SOMの更新後は `[[0.5,0], [1.75,1.5], [4,0]]` です。
超越関数のシグモイド値だけ小数6桁に丸めます（表示の丸め誤差は最大0.0000005）。

In [ ]:
baseline_result = demo_result()
for name, value in baseline_result.items():
    print(name, value)

## 実験1：しきい値だけを変更

次のセルは `changed_thresholds` の先頭を `-1` から `-2` に変更した例です。そのまま実行して基準と比較します。
入力 `(1,0)` の中間総入力が `-1 → 0`、最終出力が `1 → 0` になることを確かめます。
元の `HIDDEN_THRESHOLDS` は残します。境界0でステップ関数が1を返すことが理由です。

実行前に変わる値を予想し、実行後は基準との違いと、その理由を説明してください。

In [ ]:
changed_thresholds = (-2, 0.5)
print("基準:", forward([1, 0]))
print("変更:", forward([1, 0], hidden_thresholds=changed_thresholds))

## 実験2：学習率と同時更新

同じ初期値で学習率だけを比べます。学習率1では更新後の損失が4.5へ増えます。
次に2段の計算を確認します。`h=a*x, o=b*h`、正解4、初期値 `x=1, a=2, b=3`。
更新前の両勾配は6と4。学習率0.125で同時更新すると `a=1.25, b=2.5`、損失は0.3828125です。
これは線形な連鎖律の実験で、多層非線形ネットワークの訓練ではありません。

実行前に変わる値を予想し、実行後は基準との違いと、その理由を説明してください。

In [ ]:
for rate in (0, 0.125, 1):
    print(gradient_step(rate=rate))
print("2段:", chain_step())

## 実験3：系列の順序と前の状態

各列は初期状態0で開始します。順序を変えたときの最後の状態を比べます。
再帰係数0では、前の状態からの影響がなくなります。重みの学習は行っていません。

実行前に変わる値を予想し、実行後は基準との違いと、その理由を説明してください。

In [ ]:
for sequence in ([1, 0], [0, 1]):
    print(sequence, "→", recurrent_states(sequence))
print("再帰係数0:", recurrent_states([1, 0], recurrent_weight=0))

## 実験4：SOMの近傍を変更

地図はリストの順にA—B—Cです。勝者は入力空間の距離で選びます。
学習率0.5、近傍係数0.5なので、隣接Bに掛かる係数は0.25。近傍係数を0にするとBは動きません。
比較ごとに同じ元のベクトルを渡します。全体の学習を再現するものではなく、1回の更新です。

実行前に変わる値を予想し、実行後は基準との違いと、その理由を説明してください。

In [ ]:
sample = [1, 0]
prototypes = [[0, 0], [2, 2], [4, 0]]
print("近傍あり:", som_step(sample, prototypes))
print("勝者だけ:", som_step(sample, prototypes, neighbor_factor=0))
print("元のベクトル:", prototypes)

## 結果を言葉にする

- 固定重みでXORが出たことと、学習したことはどう違いますか。
- 勾配が正のとき、なぜ重みから学習率×勾配を引きますか。
- RNNで更新した「状態」と、学習で更新する「重み」はどう違いますか。
- SOMで勝者を選ぶ距離と、近傍を決める距離はどこで測っていますか。

来歴：固定ネットワークは昨年度 `chap05/memo.md` 表5.Aから採用し、コードを数式に合わせて修正。
それ以外の数値例は今年度作成。外部データや個人データはありません。
ボルツマンマシンの学習、RNNの訓練、SOMの反復学習、未知データでの性能評価はこの演習の対象外です。